# Module 17: API & Backend Basics

**Lesson: Building REST APIs with FastAPI for ML Model Deployment**

This notebook covers FastAPI fundamentals, Pydantic models, CRUD operations, serving ML models (with a focus on house price prediction), async endpoints, dependency injection, testing with httpx, CORS configuration, and environment management with pydantic-settings.

By the end of this lesson, you will be able to deploy a trained sklearn model as a production-ready REST API with input validation, error handling, and automatic documentation.

In [ ]:
import json
import sys
print('Python version:', sys.version)
print('Ready to build FastAPI applications for ML model serving')

## 1. FastAPI Fundamentals

FastAPI is a modern, fast web framework for building APIs with Python. It uses Python type hints for automatic request validation, serialization, and documentation generation.

**Key concepts:**
- `FastAPI()` — create the application instance
- `@app.get()`, `@app.post()`, etc. — route decorators
- Path parameters: `/items/{item_id}` — extracted from the URL path
- Query parameters: `?q=search&page=2` — extracted from the URL query string
- Request body: parsed from JSON automatically using Pydantic models
- Automatic Swagger docs at `/docs` and ReDoc at `/redoc`

In [ ]:
# Example: Defining a basic FastAPI app with various route types
# In a real project, save this as main.py and run:
#   uvicorn main:app --reload
from fastapi import FastAPI
from typing import Optional

app = FastAPI(
    title='House Price Prediction API',
    version='1.0.0',
    description='ML API for predicting house prices based on property features'
)

@app.get('/')
def root():
    return {'api': 'House Price Predictor', 'version': '1.0.0', 'status': 'running'}

@app.get('/health')
def health():
    return {'status': 'healthy', 'model_loaded': True}

# Path parameter example
@app.get('/models/{model_version}')
def get_model_info(model_version: str):
    return {'model': 'house_price_predictor', 'version': model_version}

# Query parameter example
@app.get('/search')
def search_features(q: Optional[str] = None, limit: int = 10):
    return {'query': q, 'results_limit': limit}

print('FastAPI app defined successfully')
print('App title:', app.title)
print('Available routes:')
for route in app.routes:
    if hasattr(route, 'methods'):
        print(f'  {route.methods} {route.path}')

## 2. Pydantic Models for Request/Response Validation

Pydantic models define the structure of request bodies and response data with automatic validation. For ML APIs, this is critical — bad input data can crash your model or produce meaningless predictions.

**Key Pydantic features:**
- `Field(ge=0, le=10)` — numeric constraints
- `Field(min_length=1, max_length=100)` — string constraints
- `Field(pattern=r'^...$')` — regex validation
- `@field_validator` — custom validation logic
- Nested models for complex request structures

In [ ]:
# Define Pydantic models for house price prediction
from pydantic import BaseModel, Field, field_validator
from typing import List, Tuple, Optional

# Input model: what the client sends
class HouseFeatures(BaseModel):
    bedrooms: int = Field(..., ge=1, le=10, description='Number of bedrooms')
    bathrooms: float = Field(..., ge=0.5, le=10, description='Number of bathrooms')
    sqft_living: int = Field(..., ge=100, le=10000, description='Living area in sq ft')
    sqft_lot: int = Field(..., ge=100, le=100000, description='Lot size in sq ft')
    floors: float = Field(..., ge=1.0, le=4.0, description='Number of floors')
    waterfront: bool = Field(False, description='Has waterfront view')
    condition: int = Field(..., ge=1, le=5, description='Condition rating 1-5')
    yr_built: int = Field(..., ge=1900, le=2025, description='Year built')

    @field_validator('sqft_living')
    @classmethod
    def validate_sqft_living(cls, v):
        if v < 100:
            raise ValueError('Living area must be at least 100 sq ft')
        return v

# Response model: what the API returns
class PricePrediction(BaseModel):
    predicted_price: float = Field(..., description='Predicted house price in USD')
    confidence_interval: Tuple[float, float] = Field(..., description='90% confidence interval [low, high]')
    prediction_id: str = Field(..., description='Unique prediction identifier')
    model_version: str = Field('1.0.0', description='Model version used')
    features_used: List[str] = Field(..., description='Feature names used for prediction')

# Batch request and response models
class BatchHouseRequest(BaseModel):
    samples: List[HouseFeatures] = Field(..., max_length=32, description='Batch of houses (max 32)')

class BatchHouseResponse(BaseModel):
    predictions: List[PricePrediction]
    batch_size: int

# Validate our models work
sample = HouseFeatures(
    bedrooms=3, bathrooms=2.0, sqft_living=1800, sqft_lot=5000,
    floors=1.5, waterfront=False, condition=3, yr_built=1995
)
print('Sample input validated:')
print(json.dumps(sample.model_dump(), indent=2))
print()
print('Input schema (from model_json_schema):')
schema = HouseFeatures.model_json_schema()
properties = schema.get('properties', {})
for name, props in properties.items():
    print(f'  {name}: {props.get("type")} constraints={props.get("ge", props.get("minLength", "none"))}')

### Validation in Action

Pydantic automatically rejects invalid data. Let's see what happens when a client sends bad values.

In [ ]:
# Demonstrate validation error handling
try:
    invalid = HouseFeatures(
        bedrooms=100,  # This should fail: ge=1, le=10
        bathrooms=2.0, sqft_living=1800, sqft_lot=5000,
        floors=1.5, waterfront=False, condition=3, yr_built=1995
    )
    print('Unexpected: validation passed')
except Exception as e:
    print('Validation correctly caught bad input:')
    errors = e.errors()
    for err in errors:
        field_path = '.'.join(str(loc) for loc in err['loc'])
        print(f'  Field: {field_path}')
        print(f'  Error: {err["msg"]}')
        print(f'  Type: {err["type"]}')
        print()

# Valid example
try:
    valid = HouseFeatures(
        bedrooms=4, bathrooms=2.5, sqft_living=2500, sqft_lot=8000,
        floors=2.0, waterfront=True, condition=4, yr_built=2005
    )
    print('Valid house features accepted:')
    print(f'  {valid.model_dump()}')
except Exception as e:
    print('Unexpected error:', e)

## 3. CRUD Operations with In-Memory Store

REST APIs typically implement Create, Read, Update, Delete operations. Here we use an in-memory dictionary as a simple data store to manage a registry of trained models.

**HTTP Methods:**
- `GET` — retrieve resources (read)
- `POST` — create new resources (create)
- `PUT` — replace existing resources (update)
- `DELETE` — remove resources (delete)

**Status codes:** 200 OK, 201 Created, 204 No Content, 400 Bad Request, 404 Not Found

In [ ]:
# CRUD example: managing a model registry
from fastapi import HTTPException
from datetime import datetime

class ModelMetadata(BaseModel):
    name: str = Field(..., min_length=1, max_length=50, description='Model name')
    version: str = Field(..., pattern=r'^\d+\.\d+\.\d+$', description='Semantic version')
    accuracy: float = Field(..., ge=0.0, le=1.0, description='Model accuracy score')
    created_at: str = ''

# In-memory data store
models_db: dict = {}

def create_model(meta: ModelMetadata):
    model_id = f'{meta.name}-{meta.version}'
    if model_id in models_db:
        raise HTTPException(status_code=400, detail='Model already exists')
    meta.created_at = datetime.now().isoformat()
    models_db[model_id] = meta
    return {'id': model_id, **meta.model_dump()}

def get_model(model_id: str):
    if model_id not in models_db:
        raise HTTPException(status_code=404, detail='Model not found')
    return {'id': model_id, **models_db[model_id].model_dump()}

def update_model(model_id: str, meta: ModelMetadata):
    if model_id not in models_db:
        raise HTTPException(status_code=404, detail='Model not found')
    meta.created_at = models_db[model_id].created_at
    models_db[model_id] = meta
    return {'id': model_id, **meta.model_dump()}

def delete_model(model_id: str):
    if model_id not in models_db:
        raise HTTPException(status_code=404, detail='Model not found')
    del models_db[model_id]
    return {'message': 'Model deleted', 'id': model_id}

def list_models():
    return {'models': list(models_db.keys()), 'count': len(models_db)}

# Test all CRUD operations
print('=== Testing CRUD Operations ===')

# CREATE
m1 = ModelMetadata(name='random_forest', version='1.0.0', accuracy=0.97)
print('CREATE:', create_model(m1))

m2 = ModelMetadata(name='gradient_boost', version='2.0.0', accuracy=0.98)
print('CREATE:', create_model(m2))

# READ
print('READ:', get_model('random_forest-1.0.0'))

# LIST
print('LIST:', list_models())

# UPDATE
m1_updated = ModelMetadata(name='random_forest', version='1.0.0', accuracy=0.98)
print('UPDATE:', update_model('random_forest-1.0.0', m1_updated))

# DELETE
print('DELETE:', delete_model('gradient_boost-2.0.0'))
print('LIST after delete:', list_models())

# ERROR: Get non-existent
try:
    get_model('nonexistent')
except HTTPException as e:
    print('ERROR (expected):', e.status_code, e.detail)

## 4. Serving an ML Model via API

This is where ML meets production. We'll:
1. Train a house price regression model using sklearn
2. Save it with joblib
3. Load it at FastAPI startup
4. Create a /predict endpoint that validates inputs and returns predictions

**Critical pattern:** Load the model ONCE at startup, not inside every request!

In [ ]:
# Train and save a house price prediction model
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import joblib
import os
import uuid

# Generate synthetic house price data
np.random.seed(42)
n_samples = 1000

bedrooms = np.random.randint(1, 6, n_samples)
bathrooms = np.random.uniform(1, 4, n_samples).round(1)
sqft_living = np.random.randint(500, 5000, n_samples)
sqft_lot = np.random.randint(1000, 50000, n_samples)
floors = np.random.uniform(1, 3, n_samples).round(1)
waterfront = np.random.randint(0, 2, n_samples)
condition = np.random.randint(1, 6, n_samples)
yr_built = np.random.randint(1950, 2024, n_samples)

# Price formula with noise
price = (
    sqft_living * 150 +
    bedrooms * 10000 +
    bathrooms * 8000 +
    waterfront * 150000 +
    condition * 20000 +
    (yr_built - 1950) * 500 +
    np.random.normal(0, 30000, n_samples)
)
price = np.maximum(price, 50000)  # Minimum price floor

X = np.column_stack([bedrooms, bathrooms, sqft_living, sqft_lot, floors, waterfront, condition, yr_built])
feature_names = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'condition', 'yr_built']
y = price

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print('Model trained successfully!')
print(f'  MAE: ${mae:,.2f}')
print(f'  R2 Score: {r2:.4f}')
print(f'  Features: {feature_names}')

# Save model
model_path = 'house_price_model.joblib'
joblib.dump({'model': model, 'feature_names': feature_names}, model_path)
print(f'Model saved to: {model_path}')
print(f'File size: {os.path.getsize(model_path) / 1024:.1f} KB')

### Creating the Prediction API

Now we build the FastAPI endpoints that use this model.

In [ ]:
# Build the prediction API as a standalone app
from fastapi import FastAPI, HTTPException
import uuid

pred_app = FastAPI(title='House Price Prediction API', version='1.0.0')

@pred_app.on_event('startup')
def load_model():
    """Load the trained model at application startup."""
    saved = joblib.load('house_price_model.joblib')
    pred_app.state.model = saved['model']
    pred_app.state.feature_names = saved['feature_names']
    print('Model loaded at startup')
    print(f'  Features: {saved["feature_names"]}')

def predict_price(features: HouseFeatures) -> dict:
    """Run prediction using the loaded model."""
    model = pred_app.state.model
    if model is None:
        raise HTTPException(status_code=503, detail='Model not loaded')
    try:
        X = np.array([[features.bedrooms, features.bathrooms, features.sqft_living,
                       features.sqft_lot, features.floors, int(features.waterfront),
                       features.condition, features.yr_built]])
        pred = model.predict(X)[0]
        pred = round(float(pred), 2)

        # Simple confidence interval based on model's tree variance
        tree_preds = np.array([tree.predict(X)[0] for tree in model.estimators_])
        std = tree_preds.std()
        low = round(float(pred - 1.645 * std), 2)
        high = round(float(pred + 1.645 * std), 2)

        return {
            'predicted_price': pred,
            'confidence_interval': (low, high),
            'prediction_id': str(uuid.uuid4()),
            'model_version': '1.0.0',
            'features_used': pred_app.state.feature_names
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'Prediction failed: {str(e)}')

@pred_app.post('/predict', response_model=PricePrediction)
def predict(features: HouseFeatures):
    """Predict house price from features."""
    return predict_price(features)

@pred_app.post('/predict-batch', response_model=BatchHouseResponse)
def predict_batch(batch: BatchHouseRequest):
    """Predict prices for multiple houses."""
    if len(batch.samples) > 32:
        raise HTTPException(status_code=422, detail='Maximum batch size is 32')
    predictions = [predict_price(s) for s in batch.samples]
    return BatchHouseResponse(predictions=predictions, batch_size=len(predictions))

@pred_app.get('/model-info')
def model_info():
    """Return information about the loaded model."""
    model = pred_app.state.model
    if model is None:
        raise HTTPException(status_code=503, detail='Model not loaded')
    return {
        'model_type': type(model).__name__,
        'features': pred_app.state.feature_names,
        'n_estimators': model.n_estimators,
        'max_depth': model.max_depth,
        'model_version': '1.0.0'
    }

# Simulate startup and test
load_model()
print()
print('Endpoints defined:')
print('  POST /predict         - Single house price prediction')
print('  POST /predict-batch   - Batch prediction (max 32)')
print('  GET  /model-info      - Model metadata')
print('  GET  /health          - Health check')
print('  GET  /                - Root info')

### Test the Prediction Endpoint

In [ ]:
# Test predictions using the function directly
test_house = HouseFeatures(
    bedrooms=4, bathrooms=2.5, sqft_living=2800, sqft_lot=12000,
    floors=2.0, waterfront=True, condition=4, yr_built=2010
)
result = predict_price(test_house)
print('=== Single Prediction ===')
print(f'  Predicted Price: ${result["predicted_price"]:,.2f}')
print(f'  Confidence Interval: ${result["confidence_interval"][0]:,.2f} - ${result["confidence_interval"][1]:,.2f}')
print(f'  Prediction ID: {result["prediction_id"][:8]}...')
print(f'  Model Version: {result["model_version"]}')
print(f'  Features Used: {len(result["features_used"])} features')

# Batch prediction test
batch = BatchHouseRequest(samples=[
    HouseFeatures(bedrooms=3, bathrooms=2.0, sqft_living=1800, sqft_lot=5000,
                 floors=1.5, waterfront=False, condition=3, yr_built=1995),
    HouseFeatures(bedrooms=5, bathrooms=3.0, sqft_living=3500, sqft_lot=15000,
                 floors=2.0, waterfront=True, condition=5, yr_built=2020),
    HouseFeatures(bedrooms=2, bathrooms=1.0, sqft_living=900, sqft_lot=3000,
                 floors=1.0, waterfront=False, condition=2, yr_built=1960)
])
batch_result = predict_batch(batch)
print()
print('=== Batch Prediction ===')
for i, p in enumerate(batch_result.predictions):
    print(f'  House {i+1}: ${p.predicted_price:,.2f}')
print(f'  Batch size: {batch_result.batch_size}')

## 5. Async Endpoints and Background Tasks

FastAPI supports async endpoints natively. Use async for IO-bound operations (HTTP calls, database queries, file reads) and sync for CPU-bound work (model inference).

**Rule of thumb:**
- Model predictions (CPU-bound) -> sync endpoints or `run_in_threadpool`
- Fetching external data (IO-bound) -> async endpoints
- Mixing both -> sync endpoint or async with thread pool for CPU work

In [ ]:
import asyncio
import time

# Simulate async endpoint behavior
async def fetch_external_market_data():
    """Simulate an IO-bound call to an external API."""
    await asyncio.sleep(0.2)  # Simulate network latency
    return {'median_price': 350000, 'inventory': 1200, 'market_trend': 'increasing'}

async def async_market_report():
    start = time.time()
    # Run multiple IO tasks concurrently
    results = await asyncio.gather(
        fetch_external_market_data(),
        fetch_external_market_data(),
        fetch_external_market_data()
    )
    elapsed = time.time() - start
    return {'reports': results, 'elapsed_seconds': round(elapsed, 3)}

# Demo
result = asyncio.run(async_market_report())
print('Async concurrent execution:')
print(f'  3 IO calls completed in {result["elapsed_seconds"]}s')
print(f'  (Sequential would be ~0.6s, concurrent is ~{result["elapsed_seconds"]}s)')
print()
print('Key insight: async excels at IO-bound tasks (HTTP, DB, filesystem)')
print('For CPU-bound model inference, use sync endpoints or run_in_threadpool')

### Background Tasks

FastAPI's BackgroundTasks let you run operations after returning a response — useful for logging, sending emails, or updating analytics.

In [ ]:
from fastapi import BackgroundTasks

# Simulated log writer
prediction_log = []

def log_prediction(prediction_id: str, price: float, features: dict):
    """Background task: log prediction to a store."""
    time.sleep(0.05)  # Simulate I/O
    entry = {
        'prediction_id': prediction_id,
        'price': price,
        'features': features,
        'timestamp': time.time()
    }
    prediction_log.append(entry)
    print(f'  [Background] Logged prediction {prediction_id[:8]}...')

def predict_with_logging(features: HouseFeatures, bg_tasks: BackgroundTasks):
    """Predict and schedule logging in background."""
    result = predict_price(features)
    bg_tasks.add_task(log_prediction, result['prediction_id'], result['predicted_price'], features.model_dump())
    return result

# Simulate
from fastapi import BackgroundTasks
bt = BackgroundTasks()
test_house = HouseFeatures(
    bedrooms=3, bathrooms=2.0, sqft_living=1600, sqft_lot=6000,
    floors=1.0, waterfront=False, condition=3, yr_built=1985
)
print('Making prediction with background logging...')
pred = predict_with_logging(test_house, bt)
print(f'  Prediction returned: ${pred["predicted_price"]:,.2f}')
print(f'  Log queue: {len(prediction_log)} entries (not yet processed)')
bt()  # Execute background tasks manually for demo
print(f'  Log queue after bg tasks: {len(prediction_log)} entries')
print(f'  Logged entry: {prediction_log[-1]["prediction_id"][:8]}...')

## 6. Dependency Injection

Dependency injection (DI) is FastAPI's mechanism for sharing resources across endpoints. Instead of accessing `app.state` directly, you define a `Depends()` callable that provides the resource.

**Benefits:**
- Clean separation of concerns
- Easier testing (swap real dependencies with mocks)
- Automatic lifecycle management (setup/teardown with yield)

In [ ]:
# Dependency injection examples
from fastapi import Depends

# Define dependencies (callable functions)
def get_model():
    """Provides the ML model object."""
    if not hasattr(pred_app.state, 'model') or pred_app.state.model is None:
        raise HTTPException(status_code=503, detail='Model not available')
    return pred_app.state.model

def get_config():
    """Provides application configuration."""
    return {
        'model_version': '1.0.0',
        'max_batch_size': 32,
        'enable_logging': True
    }

# Composed dependency (uses other dependencies)
def get_model_info(model=Depends(get_model), config=Depends(get_config)):
    """Provides model metadata using injected dependencies."""
    return {
        'model_type': type(model).__name__,
        'n_estimators': model.n_estimators,
        'features': pred_app.state.feature_names,
        'version': config['model_version']
    }

# Simulate using dependencies
model_instance = get_model()
config = get_config()
info = get_model_info(model_instance, config)
print('Model info from dependency injection:')
for key, value in info.items():
    print(f'  {key}: {value}')
print()
print('Tip: In a real FastAPI app, use @app.get("/info")\n'
      '      def info(model: ModelType = Depends(get_model)): ...')

### Generator Dependencies (Context Managers)

Use `yield` in a dependency to run cleanup code after the request completes.

In [ ]:
# Generator dependency pattern for resource management
class DatabaseConnection:
    """Mock database connection."""
    def __init__(self):
        self.connected = False
    def connect(self):
        self.connected = True
        print('  [DB] Connected')
    def query(self, sql):
        if not self.connected:
            raise RuntimeError('Not connected')
        return {'result': 'data', 'sql': sql}
    def close(self):
        self.connected = False
        print('  [DB] Closed')

def get_db():
    """Dependency that manages database lifecycle."""
    db = DatabaseConnection()
    db.connect()
    try:
        yield db
    finally:
        db.close()

# Simulate usage
print('Using generator dependency:')
gen = get_db()
db = next(gen)  # Acquire (before yield)
result = db.query('SELECT * FROM predictions')
print(f'  Query result: {result}')
try:
    next(gen)  # This will trigger finally block
except StopIteration:
    print('  Dependency cleanup completed')

## 7. Testing with httpx and pytest

FastAPI's TestClient lets you test endpoints without running a real server. It uses httpx under the hood and integrates perfectly with pytest.

**Testing strategy:**
- Test each endpoint with valid data (expect 200)
- Test with invalid data (expect 422)
- Test edge cases (missing resources -> 404)
- Use fixtures to create reusable test setup

In [ ]:
from fastapi.testclient import TestClient

# Create a fresh test app with minimal endpoints
test_app = FastAPI()

@test_app.get('/')
def root():
    return {'status': 'running'}

@test_app.get('/health')
def health():
    return {'status': 'healthy', 'model_loaded': True}

@test_app.post('/predict')
def predict(features: HouseFeatures):
    # Use our real model from pred_app
    return predict_price(features)

client = TestClient(test_app)

# Run tests
def run_tests():
    passed = 0
    failed = 0

    # Test 1: Root endpoint
    response = client.get('/')
    if response.status_code == 200 and response.json()['status'] == 'running':
        print('PASS: GET / returns 200 with status')
        passed += 1
    else:
        print('FAIL: GET /')
        failed += 1

    # Test 2: Health endpoint
    response = client.get('/health')
    if response.status_code == 200 and response.json()['status'] == 'healthy':
        print('PASS: GET /health returns 200')
        passed += 1
    else:
        print('FAIL: GET /health')
        failed += 1

    # Test 3: Predict with valid data
    valid_payload = {
        'bedrooms': 3, 'bathrooms': 2.0, 'sqft_living': 1800,
        'sqft_lot': 5000, 'floors': 1.5, 'waterfront': False,
        'condition': 3, 'yr_built': 1995
    }
    response = client.post('/predict', json=valid_payload)
    if response.status_code == 200:
        data = response.json()
        if all(k in data for k in ['predicted_price', 'confidence_interval', 'prediction_id']):
            print('PASS: POST /predict valid data returns 200 with expected keys')
            passed += 1
        else:
            print('FAIL: POST /predict missing keys')
            failed += 1
    else:
        print(f'FAIL: POST /predict valid data got {response.status_code}')
        failed += 1

    # Test 4: Predict with invalid data (should return 422)
    invalid_payload = {
        'bedrooms': 100, 'bathrooms': 2.0, 'sqft_living': 1800,
        'sqft_lot': 5000, 'floors': 1.5, 'waterfront': False,
        'condition': 3, 'yr_built': 1995
    }
    response = client.post('/predict', json=invalid_payload)
    if response.status_code == 422:
        print('PASS: POST /predict invalid data returns 422')
        passed += 1
    else:
        print(f'FAIL: POST /predict invalid data got {response.status_code}')
        failed += 1

    # Test 5: Missing required field
    missing_payload = {'bedrooms': 3}  # Missing many fields
    response = client.post('/predict', json=missing_payload)
    if response.status_code == 422:
        print('PASS: POST /predict missing fields returns 422')
        passed += 1
    else:
        print(f'FAIL: POST /predict missing fields got {response.status_code}')
        failed += 1

    print()
    print(f'Tests: {passed} passed, {failed} failed out of {passed + failed}')
    return passed, failed

run_tests()

## 8. CORS and Environment Configuration

### CORS (Cross-Origin Resource Sharing)

When a frontend (like React or Vue) hosted at `http://localhost:5173` calls your API at `http://localhost:8000`, the browser blocks the request unless the API explicitly allows it via CORS headers.

In [ ]:
# CORS Configuration
from fastapi.middleware.cors import CORSMiddleware

cors_app = FastAPI(title='CORS Demo')

cors_app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        'http://localhost:5173',   # Vite dev server
        'http://localhost:3000',   # React dev server
        'https://myapp.com'        # Production frontend
    ],
    allow_credentials=True,
    allow_methods=['*'],           # Allow all HTTP methods
    allow_headers=['*'],           # Allow all headers
)

print('CORS middleware configured')
print('Allowed origins:')
for origin in cors_app.user_middleware[0].options.get('allow_origins', []):
    print(f'  - {origin}')
print()
print('Note: In production, restrict origins to your actual frontend domain.')

### Environment Configuration with pydantic-settings

pydantic-settings provides type-safe environment variable management. Create a `.env` file alongside your app to override defaults.

In [ ]:
# Environment configuration with pydantic-settings
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    app_name: str = 'House Price Prediction API'
    debug: bool = False
    model_path: str = 'house_price_model.joblib'
    api_key: str = ''
    max_batch_size: int = 32
    log_level: str = 'INFO'

    class Config:
        env_file = '.env'
        env_file_encoding = 'utf-8'

# Load settings (reads from .env if it exists, otherwise uses defaults)
settings = Settings()
print('Settings loaded:')
print(f'  app_name: {settings.app_name}')
print(f'  debug: {settings.debug}')
print(f'  model_path: {settings.model_path}')
print(f'  max_batch_size: {settings.max_batch_size}')
print(f'  log_level: {settings.log_level}')
print()
print('Create a .env file to override these values:')
print('  MODEL_PATH=custom_model.joblib')
print('  DEBUG=true')
print('  MAX_BATCH_SIZE=64')

## 9. Complete Production Structure

Here's how a production ML API project is typically structured:

In [ ]:
print('''
Project Structure:
house_price_api/
├── main.py              # FastAPI app, routes, startup
├── models.py             # Pydantic request/response models
├── settings.py           # pydantic-settings configuration
├── model.py              # Model loading and prediction logic
├── train_model.py        # Training script (produces model.joblib)
├── test_api.py          # pytest tests
├── .env                 # Environment variables (gitignored)
├── requirements.txt      # Python dependencies
├── Dockerfile            # Container definition
├── README.md             # Usage instructions
└── model.joblib          # Trained model file (gitignored)
''')
print('This separation keeps concerns isolated and code maintainable.')

### Example: Production main.py

A complete, production-ready FastAPI app:

In [ ]:
print('''
# --- main.py ---
from fastapi import FastAPI, Depends, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from contextlib import asynccontextmanager
import joblib

from models import HouseFeatures, PricePrediction, BatchHouseRequest, BatchHouseResponse
from settings import Settings

settings = Settings()
model = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    """Load model on startup, clean up on shutdown."""
    global model
    model = joblib.load(settings.model_path)
    print(f'Model loaded from {settings.model_path}')
    yield
    print('Shutting down...')

app = FastAPI(title=settings.app_name, lifespan=lifespan, version='1.0.0')

app.add_middleware(
    CORSMiddleware,
    allow_origins=settings.allowed_origins.split(','),
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.get('/')
def root():
    return {'api': settings.app_name, 'version': '1.0.0', 'status': 'running'}

@app.post('/predict', response_model=PricePrediction)
def predict(features: HouseFeatures):
    if model is None:
        raise HTTPException(status_code=503, detail='Model not loaded')
    try:
        # ... prediction logic ...
        pass
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
''')

print('\nKey improvements in production version:')
print('  1. lifespan context manager (modern alternative to @app.on_event)')
print('  2. Settings from environment variables')
print('  3. CORS configured based on settings')
print('  4. Proper error handling with try/except')
print('  5. Model availability check before prediction')

## Summary

In this lesson, you learned:

**FastAPI Fundamentals**
- Creating FastAPI apps with route decorators: @app.get, @app.post, @app.put, @app.delete
- Path parameters (`/items/{id}`) and query parameters (`?q=search`)
- Automatic OpenAPI/Swagger documentation at /docs
- HTTP status codes: 200, 201, 204, 400, 404, 422, 500, 503

**Pydantic Models**
- Input validation with Field(ge=, le=, min_length=, pattern=)
- Custom validators with @field_validator
- Request/response models separate schema from logic
- Nested models for complex data structures (batch requests)

**Serving ML Models**
- Load model ONCE at startup (not in every request)
- Use POST /predict with validated Pydantic input models
- Return predictions with metadata (confidence, version, ID)
- Handle errors gracefully with HTTPException

**Async & Background Tasks**
- Async for IO-bound operations (HTTP calls, DB queries)
- Sync for CPU-bound work (model inference)
- BackgroundTasks for non-blocking post-processing (logging)

**Dependency Injection**
- Depends() for clean resource sharing
- Generator dependencies (yield) for resource cleanup
- Composable: dependencies can depend on other dependencies

**Testing & Configuration**
- TestClient for serverless integration testing
- Test valid data (200), invalid data (422), missing resources (404)
- CORS middleware for cross-origin frontend access
- pydantic-settings for type-safe environment configuration

**Key Practice Points:**
- Load model once at startup, not in every endpoint call
- Always validate inputs with Pydantic Field constraints matching training data
- Use async for IO, thread pool or sync for CPU-bound work
- Write tests for every endpoint with both valid and invalid data
- Keep configuration in environment variables, not in code
- Handle errors gracefully: return appropriate HTTP status codes